## Imports og tilkobling

In [13]:
import sys
import os
sys.path.append('../../../src')
import feature_engineering.værdata as vd
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

load_dotenv()

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_name = "vaerdata"

## Finn og velg stasjon

In [14]:
# Breive koordinater
breive_coords = (59.57624, 7.28038)

# Velg stasjon
nearest_station, stations_df = vd.finn_naermeste_stasjon(*breive_coords, stasjon_index=0)
print("Henter data fra stasjon:", nearest_station)

Henter data fra stasjon: SN40880


## Definer periode

In [15]:
vinter_periods = [
    ("2024-11-01T00:00:00Z", "2024-11-30T23:00:00Z"),
    ("2024-12-01T00:00:00Z", "2024-12-31T23:00:00Z"),
    ("2025-01-01T00:00:00Z", "2025-01-31T23:00:00Z"),
    ("2025-11-01T00:00:00Z", "2025-11-30T23:00:00Z"),
    ("2025-12-01T00:00:00Z", "2025-12-31T23:00:00Z"),
    ("2026-01-01T00:00:00Z", "2026-01-20T22:00:00Z")
]

## Hent time for time værdata

In [16]:
weather_df_hourly = vd.fetch_weather_periods_hourly(nearest_station, vinter_periods)
weather_df_hourly.head()

,timestamp,air_temperature,wind_speed,precipitation_mm
0,2024-11-01 00:00:00,7.9,8.2,0.6
1,2024-11-01 01:00:00,7.6,6.1,1.3
2,2024-11-01 02:00:00,7.6,6.9,2.2
3,2024-11-01 03:00:00,5.9,8.4,2.8
4,2024-11-01 04:00:00,4.5,6.5,1.1


## Last opp til Azure Blob

In [17]:
import io

# --- Lag CSV i minnet ---
csv_buffer = io.BytesIO()
weather_df_hourly.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)  # gå til starten av bufferet

# --- Last opp til blob ---
output_blob_name = "BREIVE_weather.csv"
blob_client = blob_service_client.get_blob_client(container=container_name, blob=output_blob_name)
blob_client.upload_blob(csv_buffer, overwrite=True)  # overwrite=True overskriver eksisterende fil

print(f"{output_blob_name} lastet opp til blob!")

BREIVE_weather.csv lastet opp til blob!
